### Lab Assignment: Commercial Data Analysis

### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: February 15, 2026

---

### INSTRUCTIONS  
In this assignment, you will work with a dataset containing information about businesses.  
Each record is a business location.  Follow the steps below, writing and running the code in blocks, and displaying the solutions.  

The path to the dataset is in a file named `find_dataset_on_rivanna.txt` in Module 3. 

Each question part is worth 1 POINT, for a total of 15 POINTS.

Hint: reaching deeper fields in json hierarchy can be done like this:  

`df.select('address.street_number')`

---

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("comm") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/18 12:17:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/18 12:18:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# note that read.json can read a zipped JSON directly

**1. (1 PT) Read in the dataset and show the number of records**

In [3]:
biz_df = spark.read.json("/standard/ds7200-apt4c/large_datasets/part-00000-a159c41a-bc58-4476-9b78-c437667f9c2b-c000.json.gz")

**2. (1 PT) Show the first 3 records**

In [4]:
biz_df.show(3)

+--------------------+--------------------+--------------------+----------------+----+--------------------+--------------------+--------------------+
|             address|       business_tags|               hours|              id|menu|             reviews|                urls|             webpage|
+--------------------+--------------------+--------------------+----------------+----+--------------------+--------------------+--------------------+
|{Woodburn, {45.15...|                NULL|                NULL|000023995a540868|NULL|                  []|{woodburn.k12.or....|{Educational Tech...|
|{Hialeah, {25.884...|{[], [{has_atm, Y...|{NULL, 1900, NULL...|0000821a1394916e|NULL|                NULL|{NULL, [yelp.com]...|                NULL|
|{Rochester, {43.1...|{[], [{accepts_cr...|{NULL, 1700, NULL...|000136e65d50c3b7|NULL|[{New (to me) qui...|{usps.com, [yelp....|{Welcome | USPS G...|
+--------------------+--------------------+--------------------+----------------+----+--------------

**3. (1 PT) Show the first 5 street addresses which are not null**  

In [5]:
biz_df.printSchema()

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- coordinates: struct (nullable = true)
 |    |    |-- lat: double (nullable = true)
 |    |    |-- lon: double (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- county: string (nullable = true)
 |    |-- full_address: string (nullable = true)
 |    |-- highway_number: string (nullable = true)
 |    |-- is_headquarters: boolean (nullable = true)
 |    |-- is_parsed: boolean (nullable = true)
 |    |-- post_direction: string (nullable = true)
 |    |-- pre_direction: string (nullable = true)
 |    |-- secondary_number: string (nullable = true)
 |    |-- state: string (nullable = true)
 |    |-- street: string (nullable = true)
 |    |-- street_address: string (nullable = true)
 |    |-- street_number: string (nullable = true)
 |    |-- street_type: string (nullable = true)
 |    |-- type_of_address: string (nullable = true)
 |    |-- zip: string (nullable = true)
 |    |-- 

In [6]:
from pyspark.sql.functions import col

In [7]:
biz_df.select('address.street_address').where(col('address.street_address').isNotNull()).show(5)

+-------------------+
|     street_address|
+-------------------+
| Cooper Contracting|
|          Route 607|
|Bush St & Kearny St|
|          S 14th St|
|             Rr 474|
+-------------------+
only showing top 5 rows


**4. (1 PT) Location**  

Count the number of records where the city is New York

In [8]:
biz_df.where(col('address.city') == "New York").count()

26/09/18 12:18:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

1953

**5. (1 PT) Hours**  

Count the number of records where closing time on Tuesday is 8pm

In [9]:
biz_df.where(col('hours.tuesday_close') == "2000").count()

3154

**6. (1 PT) Location and Hours**  

For the records where the city is New York, aggregate by Tuesday closing time, showing a column called `count` with the number of records for each Tuesday closing time. Sort by the `count` column in descending order. Here is an example of results output:

+-------------+-----+  
|tuesday_close|count|  
+-------------+-----+  
|         1800| 2001|  
|         1700| 1800|  
|         2000| 1550|  

In [10]:
(biz_df
    .where(col('address.city') == "New York")
    .where(col('hours.tuesday_close').isNotNull())
    .groupBy('hours.tuesday_close')
    .count()
    .orderBy('count', ascending=False)
    .show(truncate=False))

[Stage 9:>                                                          (0 + 1) / 1]

+-------------+-----+
|tuesday_close|count|
+-------------+-----+
|1700         |174  |
|1800         |92   |
|1900         |57   |
|2000         |48   |
|2100         |35   |
|2300         |19   |
|2200         |18   |
|0000         |15   |
|1830         |14   |
|1730         |12   |
|1600         |10   |
|1930         |8    |
|2030         |7    |
|0100         |5    |
|2330         |5    |
|2359         |5    |
|0400         |5    |
|2230         |5    |
|1500         |4    |
|1630         |4    |
+-------------+-----+
only showing top 20 rows


**7. (1 PT) Price Range**  

Price range is quoted in number of dollar signs.  Count the number of records with price range greater than or equal to three.

In [11]:
(biz_df
     .where(col('menu.price_range')
     .cast('int') >= 3)
     .count())

115

**8. (1 PT) Missing Webpage URL**  

Count the number of records that are missing the webpage url.

In [12]:
(biz_df
     .where(col('webpage.url').isNull())
     .count())

79813

**9. (1 PT) Webpage URLs**  

Register the dataframe as a temp table.  
Next, use Spark SQL to select only the webpage title column, filtering on rows where the webpage url (accessed under `webpage.url`) is *Target.com*. 

Show only one resulting row and don't truncate the output.

In [13]:
biz_df.createOrReplaceTempView("biz_tbl")

In [14]:
spark.sql("""
SELECT webpage.title
FROM biz_tbl
WHERE webpage.url = "Target.com"
""").show(1, truncate=False)

+-------------------------------+
|title                          |
+-------------------------------+
|Target : Expect More. Pay Less.|
+-------------------------------+
only showing top 1 row


**10. (1 PT) Analysis on Ratings**  

The reviews contains information such as the number of stars for each review (the *rating*).  
The ratings are stored in an array (`reviews.stars`) for each business location (you should check for yourself). Return the top five most common rating arrays.  For example, an array might look like: 
[5, 5]



In [15]:
(biz_df
    .select('reviews.stars')
    .show(5))

(biz_df
    .select('reviews.stars')
    .where(col('stars').isNotNull())
    .show(5, truncate=False))

+------+
| stars|
+------+
|    []|
|  NULL|
|[4, 4]|
|  NULL|
|  NULL|
+------+
only showing top 5 rows
+---------------------------------------------------------------------------+
|stars                                                                      |
+---------------------------------------------------------------------------+
|[]                                                                         |
|[4, 4]                                                                     |
|[5]                                                                        |
|[NULL, NULL, NULL, NULL]                                                   |
|[NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 1, 5, 2, 4, 5, 1, 4, 4, 4]|
+---------------------------------------------------------------------------+
only showing top 5 rows


In [16]:
(biz_df
    .select('reviews.stars')
    .where(col('stars').isNotNull())
    .groupBy('stars')
    .count()
    .orderBy('count', ascending=False)
    .show(5))

[Stage 21:>                                                         (0 + 1) / 1]

+------+-----+
| stars|count|
+------+-----+
|    []|42419|
|   [5]| 4258|
|[NULL]| 3067|
|[5, 5]| 1610|
|   [1]| 1559|
+------+-----+
only showing top 5 rows


**11. More work with Ratings**  

For this question, you will filter out null ratings and then compute the average rating for each business location (using the field: `id`).


a) (1 PT) Create a new dataframe retaining two fields: `id`, `reviews.stars`


In [50]:
rate_df = biz_df.select('id', 'reviews.stars')

rate_df.show(5)

+----------------+------+
|              id| stars|
+----------------+------+
|000023995a540868|    []|
|0000821a1394916e|  NULL|
|000136e65d50c3b7|[4, 4]|
|00014329a70b9869|  NULL|
|00031c0a83f00657|  NULL|
+----------------+------+
only showing top 5 rows


b) (1 PT) Create a row for each rating  
hint: use the `withColumn()` and `explode()` functions  
you will need to import the `explode()` function by issuing:

`from pyspark.sql.functions import explode`


In [21]:
from pyspark.sql.functions import explode

In [51]:
rate_df = rate_df.withColumn('stars', explode('stars'))

rate_df.show(5)

+----------------+-----+
|              id|stars|
+----------------+-----+
|000136e65d50c3b7|    4|
|000136e65d50c3b7|    4|
|0003b7589a4e12a0|    5|
|00045f958e4bb02a| NULL|
|00045f958e4bb02a| NULL|
+----------------+-----+
only showing top 5 rows


c) (1 PT) Return a count of the number of ratings in this dataframe

In [52]:
rate_df.count()

600082

d) (1 PT) Drop rows where the rating is null, and return a count of the number of non-null ratings

In [53]:
rate_df = (rate_df
               .where(col('stars').isNotNull()))

rate_df.count()

538241

e) (1 PT) Compute the average rating, grouped by `id`. After the average is computed, sort by `id` in ascending order and show the top 10 records.  
 
hint:   
this can all be done in one line using the `agg()` function  
this `id` should be at the top: 000136e65d50c3b7

In [54]:
from pyspark.sql.functions import avg

In [60]:
(rate_df
    .groupBy('id')
    .agg(avg('stars')
    .alias('Average Rating'))
    .orderBy('id')
    .show(10))

[Stage 67:>                                                         (0 + 1) / 1]

+----------------+------------------+
|              id|    Average Rating|
+----------------+------------------+
|000136e65d50c3b7|               4.0|
|0003b7589a4e12a0|               5.0|
|00059519f0dba1b4|3.3333333333333335|
|000a1df4c8e0ecd2|               4.6|
|000c7b7a30623083|               5.0|
|000c9ffc8b89af03|               3.0|
|000de20baa847ecc|1.6666666666666667|
|001064359d9f162f|               5.0|
|0010c9f495d87dd7|               3.0|
|0017774db5e6400a| 4.333333333333333|
+----------------+------------------+
only showing top 10 rows
